# 08_agents

对应 `index.md` 第 8 阶段：Agent / 工具调用闭环。

笔记本在 `codes/langchain/08_agents/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT` 并加载 `codes/langchain/local.env`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

本阶段不引入 LangGraph。所谓 Agent，本体就是**一个循环**：模型要工具 → 你执行工具 → 把结果塞回消息历史 → 再问模型，直到模型不再要工具。前几阶段（消息、工具、LCEL）在这里合流。

每题都会真的调 API，跑得比前面阶段慢。

In [1]:
import os
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "08_agents" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "langchain" / "08_agents"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

def load_local_env(env_path: Path) -> None:
    """把 local.env 写入 os.environ（已有键不覆盖）。"""
    if not env_path.is_file():
        return
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip("\"'")
        if key and key not in os.environ:
            os.environ[key] = value

ROOT = lab_root()
load_local_env(ROOT.parent / "local.env")
ROOT, bool(os.environ.get("DEEPSEEK_API_KEY"))

(PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/langchain/08_agents'),
 True)

## 1. 完成一轮闭环，拿到最终答案

已有绑定好工具的 `bound`、`tools_by_name`，以及只含一条 `HumanMessage` 的 `messages`。

按顺序做：

1. `bound.invoke(messages)` 得到 `ai`，打印它的 `tool_calls`
2. 把 `ai` **本身**追加进 `messages`
3. 执行第一条 `tool_call`，用结果构造 `ToolMessage`（`tool_call_id` 要对齐），追加进 `messages`
4. 再 `bound.invoke(messages)` 一次，打印返回的 `content`

第 2 步别漏——只放 `ToolMessage` 不放 `AIMessage` 的后果，下一题就看到了。

In [8]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b

@tool
def mul(a: int, b: int) -> int:
    """两数相乘。"""
    return a * b

tools_by_name = {"add": add, "mul": mul}

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)
bound = model.bind_tools([add, mul])

messages = [HumanMessage("2+3 等于几？必须用工具。")]

# 作答

response =bound.invoke(messages)

tool_call = response.tool_calls[0]
print(tool_call)

observe = tools_by_name[tool_call["name"]].invoke(tool_call["args"])

messages.append(response)

messages.append(ToolMessage(content=str(observe), tool_call_id=tool_call["id"]))

response = bound.invoke(messages)

print(response.content)
#评阅
# 对。四步齐全：invoke、把 AIMessage 也 append 进去（第 2 步没漏）、执行工具、tool_call_id 对齐、再 invoke 拿答案。
# 一处小点：题面说「打印它的 tool_calls」，这里只打了第一条 tool_calls[0]。
#   目前模型只返回一条，看不出差别；一旦并行调用多个工具就会漏看，想看全该 print(response.tool_calls)。
# 变量名 observe 装的是工具返回值，叫 observation 更贴切。

#参考答案
# ai = bound.invoke(messages)
# print(ai.tool_calls)
# messages.append(ai)
# call = ai.tool_calls[0]
# messages.append(ToolMessage(
#     content=str(tools_by_name[call["name"]].invoke(call["args"])),
#     tool_call_id=call["id"],
# ))
# print(bound.invoke(messages).content)

{'name': 'add', 'args': {'a': 2, 'b': 3}, 'id': 'call_00_gSw94CkacNR7DcVHmNS06693', 'type': 'tool_call'}
2 + 3 = **5**


## 2. 漏放 `AIMessage` 会怎样

上一题要求把 `ai` 也追加进 `messages`。这题把漏掉它的后果跑出来。

已有只含 `HumanMessage` 的 `messages`。这次**故意**只追加 `ToolMessage`、不追加 `ai`，然后调 `bound.invoke(messages)`。

把这次调用包进 `try/except`：出错就在 `except` 里打印异常类型与消息（`type(e).__name__`、`str(e)`）。

想清楚为什么：工具结果不能凭空出现，它必须是对某条带 `tool_calls` 的消息的回应。

In [10]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b

@tool
def mul(a: int, b: int) -> int:
    """两数相乘。"""
    return a * b

tools_by_name = {"add": add, "mul": mul}

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)
bound = model.bind_tools([add, mul])

messages = [HumanMessage("2+3 等于几？必须用工具。")]

# 作答


response =bound.invoke(messages)

tool_call = response.tool_calls[0]
print(tool_call)

observe = tools_by_name[tool_call["name"]].invoke(tool_call["args"])

# messages.append(response) # 故意漏掉
try:
    messages.append(ToolMessage(content=str(observe), tool_call_id=tool_call["id"]))
    response = bound.invoke(messages)
    print(response.content)
except Exception as e:
    print(type(e).__name__, str(e))
#评阅
# 对。故意漏掉 AIMessage，try/except 抓到了 OpenAIInvalidRequestError，异常类型与消息都打了。
# 你看到的就是原因：role='tool' 的消息必须回应一条带 tool_calls 的先行消息。
#   少了 AIMessage，ToolMessage 没有可回应的对象，服务端直接 400。
# 记住这题的重点：消息历史必须严格成对——AIMessage(tool_calls) 后面才能接 ToolMessage。

#参考答案
# messages.append(ToolMessage(
#     content=str(tools_by_name[tool_call["name"]].invoke(tool_call["args"])),
#     tool_call_id=tool_call["id"],
# ))
# try:
#     bound.invoke(messages)
# except Exception as e:
#     print(type(e).__name__, str(e))
    


{'name': 'add', 'args': {'a': 2, 'b': 3}, 'id': 'call_00_v6gVWKpEs5EIj8TJMNI61191', 'type': 'tool_call'}
OpenAIInvalidRequestError Error code: 400 - {'error': {'message': "Messages with role 'tool' must be a response to a preceding message with 'tool_calls' (request_id: 53af990e-64e7-4cac-a5ea-fd46cc190643)", 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}


## 3. 写成循环，直到模型不再要工具

单轮只能推进一步，而任务常常要多步。真正的 Agent 是个循环。

已有 `bound`、`tools_by_name`，以及 `question`（需要三步才能算完）。写一个循环，每轮：

1. `bound.invoke(messages)` 得到 `ai`，追加进 `messages`
2. 若 `ai.tool_calls` 为空 → 模型已经给出答案，跳出循环
3. 否则逐条执行每个 tool_call，各构造一条 `ToolMessage` 追加进 `messages`

循环结束后打印**一共几轮**，再打印最终答案（`messages[-1].content`）。

注意一次可能返回**多个** tool_call，所以第 3 步要遍历，不能只取第一条。

In [11]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b

@tool
def mul(a: int, b: int) -> int:
    """两数相乘。"""
    return a * b

tools_by_name = {"add": add, "mul": mul}

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)
bound = model.bind_tools([add, mul])

question = "先算 2+3，再把结果乘以 4。必须用工具。"
messages = [HumanMessage(question)]

# 作答

MAX_ITER = 10
for _ in range(MAX_ITER):
    messages.append(bound.invoke(messages))

    if messages[-1].tool_calls:
        for tool_call in messages[-1].tool_calls:
            observe = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
            messages.append(ToolMessage(content=str(observe), tool_call_id=tool_call["id"]))
    else:
        break

print(messages[-1].content)
#评阅
# 循环逻辑对，答案也算对了。但缺一半：题面要「打印一共几轮」，你没打，只打了 messages[-1].content。
#   轮数 = 模型被调用的次数。可以自己计数，也可以从消息条数反推：
#   每轮固定加 1 条 AIMessage，故轮数 = messages 里 AIMessage 的个数。
# 另两处可改：
#   1) MAX_ITER=10 是第 4 题才要的保险，这题直接 while True 更贴题；写了也不算错。
#   2) `messages.append(bound.invoke(messages))` 再靠 messages[-1] 取回，读起来绕；留个局部变量 ai 更清楚。

#参考答案
# steps = 0
# while True:
#     ai = bound.invoke(messages)
#     messages.append(ai)
#     steps += 1
#     if not ai.tool_calls:
#         break
#     for call in ai.tool_calls:
#         messages.append(ToolMessage(
#             content=str(tools_by_name[call["name"]].invoke(call["args"])),
#             tool_call_id=call["id"],
#         ))
# print(steps)
# print(messages[-1].content)
    

结果是 **20**。

计算过程：
1. 2 + 3 = 5
2. 5 × 4 = 20


## 4. 给循环装一个上限

循环有个真实风险：模型反复要工具、迟迟不给答案，就会一直转下去。生产里的 Agent 都带保险。

改成「最多 `max_steps` 轮」：正常结束就取答案；轮数用满仍未结束，就停下来记一句「达到上限、未完成」，不要继续问下去。

已有 `bound`、`tools_by_name`、`question`、`max_steps`。把结果存进 `answer` 和 `status` 两个变量（`status` 取 `"完成"` 或 `"超上限"`），打印它们。

先按写好的 `max_steps = 2` 跑一次（这个问题要 3 轮，应报超上限），再把 `max_steps` 改成 `6` 跑一次（应完成）。

In [15]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b

@tool
def mul(a: int, b: int) -> int:
    """两数相乘。"""
    return a * b

tools_by_name = {"add": add, "mul": mul}

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)
bound = model.bind_tools([add, mul])

question = "先算 2+3，再把结果乘以 4。必须用工具。"
max_steps = 2

# 作答

answer = None
status = None
messages = [HumanMessage(question)]
for _ in range(max_steps):
    messages.append(bound.invoke(messages))

    if messages[-1].tool_calls:
        for tool_call in messages[-1].tool_calls:
            observe = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
            messages.append(ToolMessage(content=str(observe), tool_call_id=tool_call["id"]))
    else:
        answer = messages[-1].content
        status = "完成"
        break
else:
    answer = "达到上限，未完成"
    status = "超上限"

print(status, answer)

answer = None
status = None
max_steps = 6
messages = [HumanMessage(question)]
for _ in range(max_steps):
    messages.append(bound.invoke(messages))

    if messages[-1].tool_calls:
        for tool_call in messages[-1].tool_calls:
            observe = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
            messages.append(ToolMessage(content=str(observe), tool_call_id=tool_call["id"]))
    else:
        answer = messages[-1].content
        status = "完成"
        break
else:
    answer = "达到上限，未完成"
    status = "超上限"

print(status, answer)
#评阅
# 对，而且 for...else 用得很地道：循环正常跑完（没 break）才走 else 的超上限分支。
# 两种情形都验证了：max_steps=2 报「超上限」，max_steps=6 报「完成」。
# 一处可改：同一段循环体贴了两遍，只为换一个 max_steps。包成函数（第 5 题）就自然消掉了。

#参考答案
# def run_once(max_steps):
#     msgs = [HumanMessage(question)]
#     for _ in range(max_steps):
#         ai = bound.invoke(msgs)
#         msgs.append(ai)
#         if not ai.tool_calls:
#             return "完成", ai.content
#         for call in ai.tool_calls:
#             msgs.append(ToolMessage(
#                 content=str(tools_by_name[call["name"]].invoke(call["args"])),
#                 tool_call_id=call["id"],
#             ))
#     return "超上限", "达到上限，未完成"
#
# status, answer = run_once(max_steps)
# print(status, answer)

超上限 达到上限，未完成
完成 结果：**20**

步骤：
1. 2 + 3 = 5
2. 5 × 4 = **20**


## 5. 封装成 `run_agent`

前面几题拼起来，就是生产里 Agent 的最小骨架。把它收成一个可复用的函数。

写 `run_agent(question: str, max_steps: int = 6) -> str`：

- 函数内部自己建 `messages = [HumanMessage(question)]`（不要依赖外面的变量）
- 跑第 4 题那样的带上限循环
- 返回最终答案字符串；若超上限，就返回一句说明文字（**不要抛异常**，调用方不该为此崩）

用下面两个问题各跑一次并打印，确认多步任务都能算对。

In [17]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b

@tool
def mul(a: int, b: int) -> int:
    """两数相乘。"""
    return a * b

tools_by_name = {"add": add, "mul": mul}

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)
bound = model.bind_tools([add, mul])

questions = [
    "先算 2+3，再把结果乘以 4。必须用工具。",
    "把 1 依次加 3、再乘 2、再减 1。必须用工具。",
]

# 作答

def run_agent(question: str, max_steps: int = 6) -> str:
    messages = [HumanMessage(question)]
    for _ in range(max_steps):
        messages.append(bound.invoke(messages))

        if messages[-1].tool_calls:
            for tool_call in messages[-1].tool_calls:
                observe = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
                messages.append(ToolMessage(content=str(observe), tool_call_id=tool_call["id"]))
        else:
            return messages[-1].content
    return "达到上限，未完成"

print(run_agent(questions[0]))
print("-"*20)
print(run_agent(questions[1]))
#评阅
# 对。签名、函数内自建 messages、返回字符串、超上限不抛异常，题面要求全满足；两问分别算出 20 和 7。
# 与第 4 题一样，messages.append(bound.invoke(...)) 再靠 [-1] 取回，能跑但绕。
# 到这一步，你已经手写出了一个可复用的 Agent 骨架。

#参考答案
# def run_agent(question: str, max_steps: int = 6) -> str:
#     msgs = [HumanMessage(question)]
#     for _ in range(max_steps):
#         ai = bound.invoke(msgs)
#         msgs.append(ai)
#         if not ai.tool_calls:
#             return ai.content
#         for call in ai.tool_calls:
#             msgs.append(ToolMessage(
#                 content=str(tools_by_name[call["name"]].invoke(call["args"])),
#                 tool_call_id=call["id"],
#             ))
#     return f"达到上限（{max_steps} 轮）仍未完成"
#
# for q in questions:
#     print(run_agent(q))


先算：2 + 3 = 5  
再算：5 × 4 = 20

结果是 **20**。
--------------------
结果是 **7**。

计算过程：
1. 1 + 3 = 4  
2. 4 × 2 = 8  
3. 8 - 1 = 7
